# IAN 603 · Module 2
# Designing Relational Databases

Turn a real-world problem into reliable tables, keys, relationships, constraints, and an analytics-ready schema.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module2.html


## Learning goals

Work through the examples in order. Focus on the meaning of each row, the reason for each SQL clause, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 2

A relational design starts with **row meaning**, then adds keys and relationships.

| Object | Grain |
|---|---|
| patient | one row per patient |
| provider | one row per provider |
| service | one row per reusable service |
| appointment | one row per appointment |
| appointment_service | one row per service used in an appointment |

### Relationship mental model

```text
patient ──< appointment >── provider
                |
                v
      appointment_service >── service
```

- A **primary key** identifies one row.
- A **foreign key** connects related rows.
- A **UNIQUE** constraint protects business identifiers.
- A **CHECK** constraint protects valid states.
- A **junction table** resolves many-to-many relationships.

Good design makes later joins and analytics much safer.


# Teaching database

Run the next cell to create or reset the Module 2 teaching data. You can return to this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 2\n-- Designing Relational Databases\n-- ============================================================\n-- Companion file for s_module2.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- This file focuses on:\n--   * tables and row grain\n--   * primary keys\n--   * foreign keys\n--   * UNIQUE and CHECK constraints\n--   * one-to-many and many-to-many relationships\n--   * a junction table\n--   * basic indexes\n--\n-- Run the schema section first, then the sample data.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - MAIN RELATIONAL SCHEMA\n-- ============================================================\n\n-- Reset the five related tables so the script can be rerun.\nDROP TABLE IF EXISTS appointment_service, appointment, service, provider, patient CASCADE;\n\n-- Parent table: one row per patient.\nCREATE TABLE patient (\n  patient_id bigint GENERATED ALWAYS AS IDENTITY,\n  mrn        text NOT NULL,\n  first_name text NOT NULL,\n  last_name  text NOT NULL,\n  birth_date date NOT NULL,\n  email      text,\n  CONSTRAINT pk_patient PRIMARY KEY (patient_id),\n  CONSTRAINT uq_patient_mrn UNIQUE (mrn),\n  CONSTRAINT uq_patient_email UNIQUE (email)\n);\n\n-- Parent table: one row per provider.\nCREATE TABLE provider (\n  provider_id   bigint GENERATED ALWAYS AS IDENTITY,\n  npi           text NOT NULL,\n  provider_name text NOT NULL,\n  specialty     text,\n  is_active     boolean NOT NULL DEFAULT TRUE,\n  CONSTRAINT pk_provider PRIMARY KEY (provider_id),\n  CONSTRAINT uq_provider_npi UNIQUE (npi),\n  CONSTRAINT chk_provider_npi CHECK (npi ~ '^[0-9]{10}$')\n);\n\n-- Parent table: one row per reusable service.\nCREATE TABLE service (\n  service_id     bigint GENERATED ALWAYS AS IDENTITY,\n  service_code   text NOT NULL,\n  service_name   text NOT NULL,\n  standard_price numeric(10,2) NOT NULL,\n  CONSTRAINT pk_service PRIMARY KEY (service_id),\n  CONSTRAINT uq_service_code UNIQUE (service_code),\n  CONSTRAINT chk_service_price CHECK (standard_price >= 0)\n);\n\n-- Event table: one row per scheduled appointment.\nCREATE TABLE appointment (\n  appointment_id bigint GENERATED ALWAYS AS IDENTITY,\n  patient_id     bigint NOT NULL,\n  provider_id    bigint NOT NULL,\n  starts_at      timestamptz NOT NULL,\n  ends_at        timestamptz NOT NULL,\n  status         text NOT NULL DEFAULT 'scheduled',\n  CONSTRAINT pk_appointment PRIMARY KEY (appointment_id),\n  CONSTRAINT fk_appointment_patient\n    FOREIGN KEY (patient_id) REFERENCES patient (patient_id) ON DELETE RESTRICT,\n  CONSTRAINT fk_appointment_provider\n    FOREIGN KEY (provider_id) REFERENCES provider (provider_id) ON DELETE RESTRICT,\n  CONSTRAINT chk_appointment_time CHECK (ends_at > starts_at),\n  CONSTRAINT chk_appointment_status CHECK (\n    status IN ('scheduled', 'completed', 'cancelled', 'no_show')\n  )\n);\n\n-- Junction table: one row per service used in one appointment.\nCREATE TABLE appointment_service (\n  appointment_id bigint NOT NULL,\n  service_id     bigint NOT NULL,\n  quantity       smallint NOT NULL DEFAULT 1,\n  unit_charge    numeric(10,2) NOT NULL,\n  CONSTRAINT pk_appointment_service PRIMARY KEY (appointment_id, service_id),\n  CONSTRAINT fk_appointment_service_appointment\n    FOREIGN KEY (appointment_id) REFERENCES appointment (appointment_id) ON DELETE CASCADE,\n  CONSTRAINT fk_appointment_service_service\n    FOREIGN KEY (service_id) REFERENCES service (service_id) ON DELETE RESTRICT,\n  CONSTRAINT chk_appointment_service_quantity CHECK (quantity > 0),\n  CONSTRAINT chk_appointment_service_charge CHECK (unit_charge >= 0)\n);\n\nSELECT table_name\nFROM information_schema.tables\nWHERE table_schema = 'public'\nORDER BY table_name;\n\n-- ============================================================\n-- SECTION 2 - SAMPLE DATA\n-- ============================================================\n\n-- Teaching reset: make this sample cell safe to rerun.\nTRUNCATE TABLE appointment_service, appointment, service, provider, patient\nRESTART IDENTITY CASCADE;\n\nINSERT INTO patient (mrn, first_name, last_name, birth_date, email)\nVALUES\n  ('MRN-1001', 'Maya', 'Brooks', DATE '1988-04-12', 'maya.brooks@example.org'),\n  ('MRN-1002', 'Evan', 'Lee', DATE '1976-11-03', NULL);\n\nINSERT INTO provider (npi, provider_name, specialty)\nVALUES\n  ('1234567890', 'Jordan Chen', 'Family Medicine'),\n  ('2345678901', 'Priya Shah', 'Internal Medicine');\n\nINSERT INTO service (service_code, service_name, standard_price)\nVALUES\n  ('S101', 'Office Visit', 95.00),\n  ('L220', 'Complete Blood Count', 28.00),\n  ('V310', 'Influenza Vaccine', 42.50);\n\nINSERT INTO appointment (patient_id, provider_id, starts_at, ends_at, status)\nSELECT p.patient_id, pr.provider_id,\n       TIMESTAMPTZ '2026-09-08 09:00:00-04',\n       TIMESTAMPTZ '2026-09-08 09:30:00-04',\n       'scheduled'\nFROM patient AS p\nCROSS JOIN provider AS pr\nWHERE p.mrn = 'MRN-1001' AND pr.npi = '1234567890';\n\nINSERT INTO appointment_service (appointment_id, service_id, quantity, unit_charge)\nSELECT a.appointment_id, s.service_id, 1, s.standard_price\nFROM appointment AS a\nJOIN patient AS p ON p.patient_id = a.patient_id\nCROSS JOIN service AS s\nWHERE p.mrn = 'MRN-1001' AND s.service_code IN ('S101', 'L220');\n\nSELECT\n  (SELECT count(*) FROM patient) AS patients,\n  (SELECT count(*) FROM provider) AS providers,\n  (SELECT count(*) FROM service) AS services,\n  (SELECT count(*) FROM appointment) AS appointments,\n  (SELECT count(*) FROM appointment_service) AS service_lines;"

execute_script(setup_sql)


# VERIFY KEYS AND RELATIONSHIPS


## VERIFY KEYS AND RELATIONSHIPS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT patient_id, mrn, first_name, last_name\nFROM patient\nORDER BY patient_id;"

display_query(query, title="VERIFY KEYS AND RELATIONSHIPS · Step 1")


## VERIFY KEYS AND RELATIONSHIPS · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT provider_id, npi, provider_name, specialty\nFROM provider\nORDER BY provider_id;"

display_query(query, title="VERIFY KEYS AND RELATIONSHIPS · Step 2")


## VERIFY KEYS AND RELATIONSHIPS · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT service_id, service_code, service_name, standard_price\nFROM service\nORDER BY service_id;"

display_query(query, title="VERIFY KEYS AND RELATIONSHIPS · Step 3")


## VERIFY KEYS AND RELATIONSHIPS · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT\n    a.appointment_id,\n    p.mrn,\n    p.first_name,\n    p.last_name,\n    pr.provider_name,\n    a.starts_at,\n    a.ends_at,\n    a.status\nFROM appointment AS a\nJOIN patient AS p\n  ON p.patient_id = a.patient_id\nJOIN provider AS pr\n  ON pr.provider_id = a.provider_id\nORDER BY a.starts_at;"

display_query(query, title="VERIFY KEYS AND RELATIONSHIPS · Step 4")


## VERIFY KEYS AND RELATIONSHIPS · Step 5

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Many-to-many relationship resolved through appointment_service.\nSELECT\n    a.appointment_id,\n    p.mrn,\n    s.service_code,\n    s.service_name,\n    aps.quantity,\n    aps.unit_charge,\n    aps.quantity * aps.unit_charge AS line_total\nFROM appointment AS a\nJOIN patient AS p\n  ON p.patient_id = a.patient_id\nJOIN appointment_service AS aps\n  ON aps.appointment_id = a.appointment_id\nJOIN service AS s\n  ON s.service_id = aps.service_id\nORDER BY a.appointment_id, s.service_code;"

display_query(query, title="VERIFY KEYS AND RELATIONSHIPS · Step 5")


# INSPECT CONSTRAINTS


## INSPECT CONSTRAINTS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT\n    tc.table_name,\n    tc.constraint_name,\n    tc.constraint_type\nFROM information_schema.table_constraints AS tc\nWHERE tc.table_schema = 'public'\n  AND tc.table_name IN (\n      'patient','provider','service','appointment','appointment_service'\n  )\nORDER BY tc.table_name, tc.constraint_type, tc.constraint_name;"

display_query(query, title="INSPECT CONSTRAINTS · Step 1")


# INDEX EXAMPLES


## INDEX EXAMPLES · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "CREATE INDEX IF NOT EXISTS idx_appointment_patient_time\n  ON appointment (patient_id, starts_at);"

execute_script(query)


## INDEX EXAMPLES · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "CREATE INDEX IF NOT EXISTS idx_appointment_provider_time\n  ON appointment (provider_id, starts_at);"

execute_script(query)


## INDEX EXAMPLES · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT indexname, indexdef\nFROM pg_indexes\nWHERE tablename = 'appointment'\nORDER BY indexname;"

display_query(query, title="INDEX EXAMPLES · Step 3")


# DESIGN VALIDATION


## DESIGN VALIDATION · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Each MRN should identify at most one patient.\nSELECT mrn, COUNT(*) AS row_count\nFROM patient\nGROUP BY mrn\nHAVING COUNT(*) > 1;"

display_query(query, title="DESIGN VALIDATION · Step 1")


## DESIGN VALIDATION · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Each appointment-service pair should be unique because the\n-- junction table uses a composite primary key.\nSELECT appointment_id, service_id, COUNT(*) AS row_count\nFROM appointment_service\nGROUP BY appointment_id, service_id\nHAVING COUNT(*) > 1;"

display_query(query, title="DESIGN VALIDATION · Step 2")


# Practice

1. State the grain of every table.
2. Identify every primary key and foreign key.
3. Explain why appointment_service needs a composite key.
4. Design a room table with:
     room_id integer PRIMARY KEY
     room_name text NOT NULL
     capacity integer CHECK (capacity > 0)
5. Add a relationship from appointments to rooms.
MODULE 2 CHECKLIST
[ ] Every table has a clear row grain.
[ ] Keys identify rows and relationships.
[ ] Constraints protect important rules.
[ ] Repeating groups are separated into related tables.
[ ] Many-to-many relationships use junction tables.

Try each task before looking back at the earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# When ready:
# display_query(practice_query, title="Practice")


# Module 2 summary

Before moving on, make sure you can explain the main ideas in your own words and can describe the **grain** or row meaning of the important query results.

A good final check is:

1. What question did the query answer?
2. What did one input row represent?
3. What does one output row represent?
4. How did SQL change the data shape or meaning?
5. How would you validate that the result is correct?

## Next
Continue to **Module 3**.
